In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
print('Размер выборки:')
df.shape
print()

print('Типы данных:')
df.info()
print()

Размер выборки:

Типы данных:
<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 86.2 KB



In [4]:
original_stats = df.describe()

In [5]:
print("Баланс классов:", df['HeartDisease'].value_counts(normalize=True).round(3))

Баланс классов: HeartDisease
1    0.553
0    0.447
Name: proportion, dtype: float64


In [6]:
np.random.seed(42)
target_size = 3000
n_to_generate = target_size - len(df)

# 1. Берём случайные существующие записи
boot_idx = np.random.randint(0, len(df), size=n_to_generate)
synth = df.iloc[boot_idx].reset_index(drop=True)

# запоминаем, где холестерин был нулём (скрытые пропуски)
zero_chol = synth['Cholesterol'] == 0

# 2. Добавляем шум 5%
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
for col in num_cols:
    std = df[col].std()
    noise = np.random.normal(0, 0.05 * std, size=n_to_generate)
    synth[col] = synth[col] + noise

# 3. Ограничения
synth.loc[zero_chol, 'Cholesterol'] = 0
synth['Cholesterol'] = synth['Cholesterol'].clip(0, 600).round(0).astype(int)

synth['Age'] = synth['Age'].clip(20, 90).round(0).astype(int)
synth['RestingBP'] = synth['RestingBP'].clip(80, 200).round(0).astype(int)
synth['MaxHR'] = synth['MaxHR'].clip(60, 210).round(0).astype(int)
synth['Oldpeak'] = synth['Oldpeak'].clip(0, 6).round(1)

# Округление
synth['FastingBS'] = synth['FastingBS'].round(0).astype(int)
synth['HeartDisease'] = synth['HeartDisease'].round(0).astype(int)

# 4. Объединяем
df = pd.concat([df, synth], ignore_index=True)
print(f"Размер датасета: {len(df)} (было 918, добавили {n_to_generate})")
print("Баланс классов:", df['HeartDisease'].value_counts(normalize=True).round(3))

Размер датасета: 3000 (было 918, добавили 2082)
Баланс классов: HeartDisease
1    0.534
0    0.466
Name: proportion, dtype: float64


In [7]:
# сравнение статистики оригинала и расширенного датасета
stats = pd.DataFrame({
    'original_mean': original_stats.loc['mean', num_cols],
    'extended_mean': df[num_cols].mean(),
})
print(stats.round(2))

             original_mean  extended_mean
Age                  53.51          53.40
RestingBP           132.40         132.68
Cholesterol         198.80         198.70
MaxHR               136.81         137.25
Oldpeak               0.89           0.90


In [8]:
df.to_csv('../data/heart_synth.csv', index=False)